In [1]:
import os
import pandas as pd

input_path = '../dataset/input-data.csv'
df = pd.read_csv(input_path)

In [17]:
import torch
import pandas as pd

def generate_summaries(tokenizer, model, df, out_path, is_gpt=False, is_llama=False, bias=None):

    if not is_llama:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model.to(device)

    ids = []
    summaries = [[] for _ in range(3)]  # generate 3 summaries per article

    for idx, row in df.iterrows():
        id, text = row.id, row.text

        print(f'{idx}: {id}')

        prompt = f'Summarize this article{" with " + bias + " bias" if bias else ""}: {text}'
        end_prompt = '\nSummary:'

        # llama allows 2048 tokens
        if is_llama:
            # llama does not need a separate tokenizer object
            tokens = model.tokenize(prompt.encode('utf-8'), add_bos=True)
            end_prompt_len = len(model.tokenize(end_prompt.encode('utf-8')))

            # truncate to context size, accounting for max summary size
            prompt = model.detokenize(tokens[:model.n_ctx()-512-end_prompt_len]).decode('utf-8')
            prompt += end_prompt

            print(prompt)
            print(len(model.tokenize(prompt.encode('utf-8'))))
            
            for i in range(len(summaries)):
                output = model(
                    prompt,
                    max_tokens=512,
                    top_p=0.95,
                    top_k=50,
                    echo=False,
                )

                summary = output['choices'][0]['text']
                print(summary)
                summaries[i].append(summary)
        else:
            offset = 0
            if hasattr(tokenizer, 'model_max_length'):
                offset = 512 if is_gpt else 0
                end_prompt_len = tokenizer(end_prompt, return_tensors="pt").input_ids.shape[1]
                inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=tokenizer.model_max_length-offset-end_prompt_len)
                offset = inputs.input_ids.shape[1] if is_gpt else 0
            else:
                prompt += end_prompt
                inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

            inputs = inputs.to(device)

            summary_ids = model.generate(
                inputs.input_ids,
                min_length=10,
                max_new_tokens=512,
                do_sample=True,
                top_k=50,
                top_p=0.95,
                num_return_sequences=len(summaries),
                pad_token_id=tokenizer.eos_token_id
            )

            for i, summary_id in enumerate(summary_ids):
                summary = tokenizer.decode(summary_id[offset:], skip_special_tokens=True)
                print(summary)
                summaries[i].append(summary)

        ids.append(id)

        # save every 30 articles
        if (idx + 1) % 30 == 0:
            save_summaries(ids, summaries, out_path)

def save_summaries(ids, summaries, out_path):
    data = {"id": ids, **{f"summary{i+1}": summaries[i] for i in range(len(summaries))}}
    df = pd.DataFrame(data)
    df.to_csv(out_path, index=False)

def summarize(tokenizer, model, df, out_path, is_gpt=False, is_llama=False):
    return generate_summaries(tokenizer, model, df, out_path, is_gpt, is_llama)

def summarize_with_leaning(tokenizer, model, df, out_path, bias, is_gpt=False, is_llama=False):
    return generate_summaries(tokenizer, model, df, out_path, is_gpt, is_llama, bias)

# BART

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration

bart_tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')
bart_model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')

In [ ]:
summarize(bart_tokenizer, bart_model, df, '../dataset/bart.csv')

In [ ]:
for leaning in ['left', 'center', 'right']:
    output_path = f'../dataset/bart-{leaning}.csv'
    summarize_with_leaning(bart_tokenizer, bart_model, df, output_path, bias=leaning)

# T5

In [ ]:
from transformers import AutoTokenizer, AutoModelWithLMHead

t5_tokenizer = AutoTokenizer.from_pretrained('t5-base')
t5_model = AutoModelWithLMHead.from_pretrained('t5-base', return_dict=True)

In [ ]:
summarize(t5_tokenizer, t5_model, df, '../dataset/t5.csv')

# GPT2 and GPT Neo

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2')

In [ ]:
summarize(gpt2_tokenizer, gpt2_model, df, '../dataset/gpt2.csv', is_gpt=True)

In [ ]:
from transformers import GPTNeoForCausalLM, GPT2Tokenizer

neo_model = GPTNeoForCausalLM.from_pretrained("EleutherAI/gpt-neo-1.3B")
neo_tokenizer = GPT2Tokenizer.from_pretrained("EleutherAI/gpt-neo-1.3B")

In [ ]:
summarize(neo_model, neo_tokenizer, df, '../dataset/neo-test.csv', is_gpt=True)

# Llama 2 (TODO)

In [ ]:
from transformers import LlamaForCausalLM, LlamaTokenizer

llama_tokenizer = LlamaTokenizer.from_pretrained("/output/path")
llama_model = LlamaForCausalLM.from_pretrained("/output/path")

Using `llama.cpp` (with llama-cpp-python bindings) to run this faster. 
- https://github.com/ggerganov/llama.cpp 
- https://github.com/abetlen/llama-cpp-python

In [3]:
import llama_cpp

In [4]:
llama_path = "path/to/llama"

In [5]:
llm = llama_cpp.Llama(model_path=llama_path, n_ctx=2048)

llama_model_loader: loaded meta data with 16 key-value pairs and 291 tensors from /Users/ellieyhc/Documents/Research/llama.cpp/models/llama-2-7b/ggml-model-q4_0.gguf (version GGUF V3 (latest))
llama_model_loader: - tensor    0:                token_embd.weight q4_0     [  4096, 32000,     1,     1 ]
llama_model_loader: - tensor    1:               output_norm.weight f32      [  4096,     1,     1,     1 ]
llama_model_loader: - tensor    2:                    output.weight q6_K     [  4096, 32000,     1,     1 ]
llama_model_loader: - tensor    3:              blk.0.attn_q.weight q4_0     [  4096,  4096,     1,     1 ]
llama_model_loader: - tensor    4:              blk.0.attn_k.weight q4_0     [  4096,  4096,     1,     1 ]
llama_model_loader: - tensor    5:              blk.0.attn_v.weight q4_0     [  4096,  4096,     1,     1 ]
llama_model_loader: - tensor    6:         blk.0.attn_output.weight q4_0     [  4096,  4096,     1,     1 ]
llama_model_loader: - tensor    7:            blk.0

In [6]:
output = llm(
  "Q: Name the planets in the solar system? A: ", # Prompt
  max_tokens=32, # Generate up to 32 tokens
  stop=["Q:", "\n"], # Stop generating just before the model would generate a new question
  echo=False # Echo the prompt back in the output
)


llama_print_timings:        load time =     817.67 ms
llama_print_timings:      sample time =       2.46 ms /    28 runs   (    0.09 ms per token, 11400.65 tokens per second)
llama_print_timings: prompt eval time =     817.55 ms /    15 tokens (   54.50 ms per token,    18.35 tokens per second)
llama_print_timings:        eval time =    2553.66 ms /    27 runs   (   94.58 ms per token,    10.57 tokens per second)
llama_print_timings:       total time =    3410.87 ms


In [7]:
print(output)

{'id': 'cmpl-41a7a333-6969-47dc-a15b-c6c98afc64fd', 'object': 'text_completion', 'created': 1700763221, 'model': '/Users/ellieyhc/Documents/Research/llama.cpp/models/llama-2-7b/ggml-model-q4_0.gguf', 'choices': [{'text': '8 (Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus and Neptune).', 'index': 0, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 15, 'completion_tokens': 28, 'total_tokens': 43}}


In [18]:
summarize(None, llm, df, 'llama.csv', is_llama=True)

0: 11c9f251-35c8-4452-b052-df9be61464c6_2
Summarize this article: Call it Cheney versus Cheney.
Mary Cheney, one of ex-Vice President Dick Cheney’s two daughters, has taken to Facebook to blast her older sibling, Elizabeth, a Wyoming Senate candidate, for the latter’s stance on same-sex marriage, The New York Times is reporting.
Mary Cheney, openly lesbian and married to Heather Poe since 2012, reportedly posted to her personal page on the social media site: “For the record, I love my sister, but she is dead wrong on the issue of marriage.
“Freedom means freedom for everyone.
That means that all families — regardless of how they look or how they are made — all families are entitled to the same rights, privileges and protections as every other.”
The Times reports Liz Cheney on Friday first articulated her position on the controversial subject, saying it should be something for voters to decide on a state-by-state basis, and not a matter for “judges” or “legislators.”
“I am not pro-gay m

Llama.generate: prefix-match hit


 Mary Cheney (who is gay) is blasting her older sister Liz Cheney for opposing same sex marriages.
Read more here: http://www.miamiherald.com/2013/08/07/v-fullstory/3549605/cheney-vs-cheney-daughters-clash-on.html#storylink=cpy



llama_print_timings:        load time =     817.67 ms
llama_print_timings:      sample time =       8.51 ms /    92 runs   (    0.09 ms per token, 10815.89 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =    8560.59 ms /    92 runs   (   93.05 ms per token,    10.75 tokens per second)
llama_print_timings:       total time =    8691.33 ms
Llama.generate: prefix-match hit


 It's not about homosexuality (or heterosexuality), it's about who is responsible for defining marriage law. The state legislatures are not authorized to change federal laws and that is what the Supreme Court ruled in 1948, Windsor v. United States. They will soon be forced to rule again on this issue because there are so many other court cases involving issues of religious liberty concerning those who oppose same-sex marriage based on their faiths.
Labels: Liz Cheney, Mary Cheney



llama_print_timings:        load time =     817.67 ms
llama_print_timings:      sample time =      10.64 ms /   115 runs   (    0.09 ms per token, 10807.25 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   11712.28 ms /   115 runs   (  101.85 ms per token,     9.82 tokens per second)
llama_print_timings:       total time =   11878.95 ms
Llama.generate: prefix-match hit


 Liz vs. Liz.
1: 84678e18-0b79-4570-9c9f-cb1cf88d968e_1
Summarize this article: The IRS official who refused to testify at a House hearing Wednesday has become a key focus of the congressional investigations into the IRS practice of singling out conservative groups.
Now under the protection of her lawyers and the Fifth Amendment, Lois Lerner is facing a maelstrom of controversy.
Members of Congress are calling her evasive, and question why she didn't reveal the program sooner -- plus her history at the Federal Elections Commission is coming under scrutiny.
Lerner touched off the public controversy when, at an American Bar Association conference earlier this month, she apologized for the IRS' practice of targeting conservative organizations for additional scrutiny.
It was the first time the agency acknowledged the practice.
"They used names like Tea Party or patriots ... and they selected cases simply because the application had those names in the title," she admitted.
She said she hadn


llama_print_timings:        load time =     817.67 ms
llama_print_timings:      sample time =       0.77 ms /     8 runs   (    0.10 ms per token, 10430.25 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =     858.21 ms /     8 runs   (  107.28 ms per token,     9.32 tokens per second)
llama_print_timings:       total time =     869.36 ms
Llama.generate: prefix-match hit


 The U.S. Supreme Court has agreed to hear a case that could determine whether the government can force Americans to buy health insurance under ObamaCare -- and it's being seen as a key test of federal power.
The justices will decide in the 2014-15 term whether the individual mandate is constitutional or not, but they also agreed to hear the case on the broader question of whether Congress can use its powers under the U.S. Constitution to require individuals to buy health insurance.
"The government's power to commandeer private parties into service of a public interest does not stretch to the point where it can compel some citizens to become active in the market for health insurance," lawyers for the plaintiffs in the case wrote in court filings. "If it did, no principled line would be left to distinguish between that power and general conscription of American citizens."
The challenge against ObamaCare came from 26 states led by Texas Attorney General Greg Abbott, as well as the Nation


llama_print_timings:        load time =     817.67 ms
llama_print_timings:      sample time =      47.30 ms /   512 runs   (    0.09 ms per token, 10823.38 tokens per second)
llama_print_timings: prompt eval time =   40376.24 ms /   700 tokens (   57.68 ms per token,    17.34 tokens per second)
llama_print_timings:        eval time =   55065.37 ms /   511 runs   (  107.76 ms per token,     9.28 tokens per second)
llama_print_timings:       total time =   96268.82 ms
Llama.generate: prefix-match hit


In [ ]:
for leaning in ['left', 'center', 'right']:
    output_path = f'../dataset/llama-{leaning}.csv'
    summarize_with_leaning(llama_tokenizer, llama_model, df, output_path, bias=leaning)